# Food Craft — Unsloth Llama-3.2-11B-Vision Fine-Tuning & Hugging Face Push

**Works on both Google Colab (`T4 GPU`) and Kaggle Notebooks (`GPU T4 x2` / `P100`)!**
1. Select a **GPU runtime** (`Runtime -> Change runtime type -> T4 GPU` on Colab, or `Accelerator -> GPU T4 x2` on Kaggle).
2. Add `HF_TOKEN` (with **Write** access under `pavankumarpalla`) to Colab Secrets or Kaggle Secrets.


In [ ]:
import os
for loader in ('colab', 'kaggle'):
    try:
        if loader == 'colab':
            from google.colab import userdata
            tok = userdata.get('HF_TOKEN')
        else:
            from kaggle_secrets import UserSecretsClient
            tok = UserSecretsClient().get_secret('HF_TOKEN')
        if tok:
            os.environ['HF_TOKEN'] = tok.strip()
            break
    except Exception:
        pass
assert os.environ.get('HF_TOKEN'), 'Set HF_TOKEN with Write permissions in Colab/Kaggle Secrets before running!'
!nvidia-smi

In [ ]:
!git clone https://github.com/PavanKumarPalla/FoodCraft.git || (cd FoodCraft && git pull)
%cd FoodCraft

In [ ]:
!python finetune_workspace/install_deps.py

In [ ]:
# Verify GPU VRAM, disk space, and HF_TOKEN write permissions BEFORE downloading or training
!python finetune_workspace/train_and_push.py --preflight-only --push-merged

In [ ]:
# Fetch pavankumarpalla/FoodCraft + Food Craft image datasets via HF_TOKEN.
# --prune-raw deletes temporary .parquet files after extraction so Colab/Kaggle disk stays safe.
# Add 'vegfru' at the end if your VM has >=65 GB free disk.
!python finetune_workspace/prepare_dataset.py --prune-raw --max-per-source 10000 --stages indian_foods_15 foodcraft_recipes foodseg103 indian_food_20 food121 food_and_vegetables food_ingredients

In [ ]:
# Fine-tune Llama-3.2-11B-Vision and push LoRA + merged model shards to Hugging Face
!python finetune_workspace/train_and_push.py --max-steps 800 --push-merged